In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
"""Pipeline INMET unificado (2016-2026) - base diária por estação.

Lê TODOS os arquivos horários do INMET da pasta bronze (qualquer ano, qualquer
variação de layout) e gera UMA base diária com as mesmas colunas:

    DATA ; PRECIPITACAO_TOTAL_MM ; TEMPERATURA_MEDIA_C ; UMIDADE_RELATIVA_MEDIA

Variações de layout tratadas automaticamente (por arquivo):
  - separador  : ';'  ou  ','  ou TAB
  - encoding   : utf-8 / latin-1
  - data       : 2016-01-01 | 2020/02/28 | 01/01/2016 | 01-01-2016
  - hora       : 00:00 | 0000 UTC | Hora UTC | HORA (UTC)   (não é usada no
                 cálculo diário, só a DATA)
  - cabeçalho  : 'Data' / 'DATA (YYYY-MM-DD)' / 'Data Medicao', precedido ou
                 não pelo bloco de metadados (REGIAO:, UF:, ... DATA DE FUNDACAO:)
  - ausentes   : -9999, vazio, NULL...
"""

import csv
import logging
import os
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

# --- 1. CONECTAR O GOOGLE DRIVE (só quando estiver no Colab) ---
try:
  from google.colab import drive

  drive.mount('/content/drive')
except ImportError:
  pass

# Configuração de Logs
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s'
)

# Caminhos no Google Drive
_BASE = Path(
    os.environ.get(
        'INMET_DIRETORIO_BASE', '/content/drive/MyDrive/projeto_escorpioes/inmet'
    )
)
PASTA_BRONZE = _BASE / '1_bronze'
ARQUIVO_SILVER = _BASE / '2_silver' / 'inmet_diario_2016_2026.csv'

# Chaves: Nome oficial final -> trechos buscados na coluna do INMET.
# Testados NA ORDEM (do mais específico ao mais genérico), sem acento/caixa.
MAPEAMENTO_COLUNAS = {
    'PRECIPITACAO_TOTAL_MM': ['PRECIPITACAO TOTAL, HORARIO', 'PRECIPITACAO TOTAL',
                              'PRECIPITACAO'],
    'TEMPERATURA_MEDIA_C': ['TEMPERATURA DO AR - BULBO SECO', 'TEMPERATURA DO AR'],
    'UMIDADE_RELATIVA_MEDIA': ['UMIDADE RELATIVA DO AR, HORARIA',
                               'UMIDADE RELATIVA DO AR', 'UMIDADE RELATIVA'],
}
# Colunas com estes termos nunca são escolhidas (máx./mín./orvalho...)
TERMOS_EXCLUIR = ('MAX', 'MIN', 'ORVALHO', 'SENSACAO')


def _soma_ou_nan(s: pd.Series) -> float:
  """Soma dos valores válidos; dia sem nenhuma medição -> NaN (não 0)."""
  return s.sum(min_count=1)


# Regras de Agregação Diária
REGRAS_AGREGACAO = {
    'PRECIPITACAO_TOTAL_MM': _soma_ou_nan,  # Soma para precipitação diária
    'TEMPERATURA_MEDIA_C': 'mean',  # Média para temperatura
    'UMIDADE_RELATIVA_MEDIA': 'mean',  # Média para umidade
}

VALORES_AUSENTES = {'', '-9999', '-9999.0', 'NULL', 'NAN', 'NA', 'N/A', '-'}

DELIMITADORES = (';', ',', '\t')


def normalizar_texto(texto) -> str:
  if texto is None or (isinstance(texto, float) and np.isnan(texto)):
    return ''
  s = unicodedata.normalize('NFKD', str(texto))
  s = ''.join(c for c in s if not unicodedata.combining(c))
  return re.sub(r'\s+', ' ', s).strip().upper()


def _eh_cabecalho(celulas) -> bool:
  """Linha de cabeçalho real: 1ª célula 'DATA...' (sem ':'), e traz colunas de
  medição. A linha 'DATA DE FUNDACAO:' do bloco de metadados NÃO conta."""
  if not celulas:
    return False
  primeira = normalizar_texto(celulas[0])
  if not primeira.startswith('DATA') or ':' in primeira:
    return False
  resto = ' | '.join(normalizar_texto(c) for c in celulas[1:])
  return any(k in resto for k in ('HORA', 'PRECIPITACAO', 'TEMPERATURA', 'UMIDADE'))


def _ler_linhas_csv(caminho: Path):
  """Lê o arquivo testando encoding e separador até achar o cabeçalho real."""
  for enc in ('utf-8-sig', 'latin-1'):
    try:
      with open(caminho, newline='', encoding=enc) as f:
        texto = f.read()
    except UnicodeDecodeError:
      continue
    for delim in DELIMITADORES:
      linhas = [r for r in csv.reader(texto.splitlines(), delimiter=delim) if r]
      if any(_eh_cabecalho(l) for l in linhas[:60]):
        return linhas, delim
  raise ValueError(
      f"{caminho.name}: Cabeçalho 'Data' não encontrado (testados separadores"
      f' {DELIMITADORES}).'
  )


def ler_arquivo_bruto(caminho: Path):
  """Devolve (DataFrame com cabeçalho correto, código da estação)."""
  ext = caminho.suffix.lower()
  if ext in ('.xlsx', '.xlsm', '.xls'):
    bruto = pd.read_excel(caminho, header=None, dtype=object)
    linhas = bruto.fillna('').astype(str).values.tolist()
  elif ext in ('.csv', '.txt'):
    linhas, _ = _ler_linhas_csv(caminho)
  else:
    raise ValueError(f'Formato não suportado: {caminho}')

  largura = max(len(l) for l in linhas)
  bruto = pd.DataFrame(
      [l + [None] * (largura - len(l)) for l in linhas], dtype=object
  )

  idx = next(
      (i for i in range(min(len(bruto), 60)) if _eh_cabecalho(list(bruto.iloc[i]))),
      None,
  )
  if idx is None:
    raise ValueError(f"{caminho.name}: Cabeçalho 'Data' não encontrado.")

  # Código da estação (metadado 'CODIGO (WMO):')
  estacao = None
  for i in range(idx):
    if normalizar_texto(bruto.iat[i, 0]).startswith('CODIGO'):
      estacao = str(bruto.iat[i, 1]).strip()

  df = bruto.iloc[idx + 1 :].copy()
  df.columns = [str(c).strip() for c in bruto.iloc[idx]]
  df = df.dropna(how='all')
  return df.reset_index(drop=True), estacao


def parse_datas(serie: pd.Series) -> pd.Series:
  """2016-01-01 | 2020/02/28 | 01/01/2016 | 01-01-2016 -> datetime."""
  s = serie.astype(str).str.strip().str[:10]
  ano_primeiro = s.str.match(r'^\d{4}[/-]\d{2}[/-]\d{2}$')
  out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
  out[ano_primeiro] = pd.to_datetime(
      s[ano_primeiro].str.replace('/', '-'), format='%Y-%m-%d', errors='coerce'
  )
  out[~ano_primeiro] = pd.to_datetime(
      s[~ano_primeiro].str.replace('-', '/'),
      format='%d/%m/%Y',
      errors='coerce',
  )
  return out


def converter_numerico(serie: pd.Series) -> pd.Series:
  s = serie.astype(str).str.strip()
  s = s.where(~s.str.upper().isin(VALORES_AUSENTES), np.nan)
  num = pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')
  return num.mask(num <= -9000)


def localizar_coluna(colunas, padroes):
  for padrao in padroes:
    p = normalizar_texto(padrao)
    for c in colunas:
      c_norm = normalizar_texto(c)
      if p in c_norm and not any(t in c_norm for t in TERMOS_EXCLUIR):
        return c
  return None


def processar_arquivo_inmet(caminho: Path) -> pd.DataFrame:
  df, estacao = ler_arquivo_bruto(caminho)

  col_data_orig = next(
      (c for c in df.columns if normalizar_texto(c).startswith('DATA')), None
  )
  if not col_data_orig:
    raise ValueError(f'Coluna Data ausente em {caminho.name}')

  out = pd.DataFrame({'DATA': parse_datas(df[col_data_orig])})

  for col_saida, padroes in MAPEAMENTO_COLUNAS.items():
    col_orig = localizar_coluna(df.columns, padroes)
    if col_orig:
      out[col_saida] = converter_numerico(df[col_orig])
      if out[col_saida].notna().sum() == 0:
        logging.warning(
            f'{caminho.name}: "{col_orig}" sem nenhum valor numérico válido.'
        )
    else:
      out[col_saida] = np.nan
      logging.error(
          f'{caminho.name}: nenhuma coluna para {col_saida}. Colunas do'
          f' arquivo: {[str(c) for c in df.columns]}'
      )

  invalidas = out['DATA'].isna().sum()
  if invalidas:
    logging.warning(f'{caminho.name}: {invalidas} linha(s) com data inválida.')
  out = out.dropna(subset=['DATA'])

  # Agregação Diária (Soma para chuva, Média para temperatura e umidade)
  diario = out.groupby('DATA', as_index=False).agg(REGRAS_AGREGACAO)

  # Arredondamento adequado
  diario['PRECIPITACAO_TOTAL_MM'] = diario['PRECIPITACAO_TOTAL_MM'].round(1)
  diario['TEMPERATURA_MEDIA_C'] = diario['TEMPERATURA_MEDIA_C'].round(1)
  diario['UMIDADE_RELATIVA_MEDIA'] = diario['UMIDADE_RELATIVA_MEDIA'].round(0)

  diario.attrs['estacao'] = estacao
  return diario


def consolidar(dfs: list[pd.DataFrame]) -> pd.DataFrame:
  """Junta os diários de todos os anos numa base única, sem duplicar datas."""
  base = pd.concat(dfs, ignore_index=True)
  # Em datas repetidas (arquivos sobrepostos) fica a linha mais completa
  base['_preenchidos'] = base[list(REGRAS_AGREGACAO)].notna().sum(axis=1)
  repetidas = int(base.duplicated(subset=['DATA']).sum())
  if repetidas:
    logging.warning(
        f'{repetidas} data(s) em mais de um arquivo; mantida a linha mais'
        ' completa de cada uma.'
    )
  base = (
      base.sort_values(['DATA', '_preenchidos'], ascending=[True, False])
      .drop_duplicates(subset=['DATA'])
      .drop(columns='_preenchidos')
      .reset_index(drop=True)
  )
  return base


def relatorio_cobertura(base: pd.DataFrame):
  """Mostra dias por ano, % de preenchimento e buracos no calendário."""
  datas = pd.to_datetime(base['DATA'])
  por_ano = datas.dt.year.value_counts().sort_index()
  logging.info('Cobertura (dias com registro por ano):')
  for ano, n in por_ano.items():
    logging.info(f'   {ano}: {n} dias')

  calendario = pd.date_range(datas.min(), datas.max(), freq='D')
  faltando = calendario.difference(datas)
  if len(faltando):
    anos_sem = sorted(set(faltando.year) - set(por_ano.index))
    logging.warning(
        f'{len(faltando)} dia(s) sem registro entre {datas.min():%Y-%m-%d} e'
        f' {datas.max():%Y-%m-%d}'
        + (f'; anos inteiros sem dados: {anos_sem}' if anos_sem else '')
    )

  for col in REGRAS_AGREGACAO:
    pct = base[col].notna().mean() * 100
    logging.info(f'   {col}: {pct:.1f}% dos dias com dado')
    if pct == 0:
      logging.error(f'{col} saiu 100% vazia - verifique os arquivos de origem.')


def executar_pipeline(pasta_entrada: Path, arquivo_saida: Path):
  arquivos = sorted(
      p
      for p in pasta_entrada.glob('**/*')
      if p.suffix.lower() in ('.csv', '.xls', '.xlsx')
      and not p.name.startswith('~')
  )
  logging.info(f'Encontrados {len(arquivos)} arquivos na pasta {pasta_entrada}.')

  dfs, estacoes = [], set()
  for arq in arquivos:
    try:
      df_proc = processar_arquivo_inmet(arq)
      dfs.append(df_proc)
      if df_proc.attrs.get('estacao'):
        estacoes.add(df_proc.attrs['estacao'])
      logging.info(
          f' Sucesso: {arq.name} ({len(df_proc)} dias, de'
          f' {df_proc["DATA"].min():%Y-%m-%d} a {df_proc["DATA"].max():%Y-%m-%d})'
      )
    except Exception as e:
      logging.error(f' Erro ao processar {arq.name}: {e}')

  if not dfs:
    logging.error('Nenhum arquivo foi processado com sucesso.')
    return

  if len(estacoes) > 1:
    logging.warning(
        f'Mais de uma estação na pasta: {sorted(estacoes)}. Datas repetidas'
        ' entre estações NÃO são comparáveis - separe por estação.'
    )

  df_consolidado = consolidar(dfs)
  df_consolidado['DATA'] = df_consolidado['DATA'].dt.strftime('%Y-%m-%d')
  relatorio_cobertura(df_consolidado)

  arquivo_saida.parent.mkdir(parents=True, exist_ok=True)
  df_consolidado.to_csv(
      arquivo_saida, sep=';', index=False, encoding='utf-8-sig'
  )
  logging.info(
      f' Pipeline concluída! Base Silver salva em: {arquivo_saida} com'
      f' {len(df_consolidado)} registros diários.'
  )
  return df_consolidado


def unir_bases_prontas(*caminhos_csv, arquivo_saida: Path):
  """Une bases diárias JÁ geradas (ex.: 2016-2018 + 2020-2026), sem duplicar
  datas. Use só com bases geradas por este código (a antiga 2016-2018 zerada
  precisa ser refeita antes)."""
  partes = []
  for c in caminhos_csv:
    d = pd.read_csv(c, sep=';', encoding='utf-8-sig')
    d['DATA'] = pd.to_datetime(d['DATA'])
    partes.append(d)
  base = consolidar(partes)
  base['DATA'] = base['DATA'].dt.strftime('%Y-%m-%d')
  relatorio_cobertura(base)
  arquivo_saida.parent.mkdir(parents=True, exist_ok=True)
  base.to_csv(arquivo_saida, sep=';', index=False, encoding='utf-8-sig')
  return base


# --- EXECUTAR PIPELINE ---
if __name__ == '__main__':
  executar_pipeline(PASTA_BRONZE, ARQUIVO_SILVER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
